# Integração das Bases de Dados
Neste notebook, faremos o passo a passo para unificar as bases de dados conforme especificado no case:

1. **Agregação da base do Bolsa Família**: vamos agregar os meses por município (código IBGE) trazendo a média das demais variáveis.
2. **Join das tabelas**: vamos fazer o join correto entre as tabelas SIDRA e Bolsa Família usando a tabela `cod_municipios` para cruzar o código de 6 dígitos com o de 7 dígitos.
3. **Exportação da base final**: salvaremos os resultados como `base_final.csv`.

In [1]:
import pandas as pd

### 1. Leitura das Bases de Dados
A leitura dos arquivos em CSV da pasta `data/`.

In [2]:
df_bf = pd.read_csv('data/bolsa_familia.csv')
df_sidra = pd.read_csv('data/sidra.csv')
df_cod = pd.read_csv('data/cod_municipios.csv', sep=';', encoding='utf-8')
import os
if os.path.exists('data/censo_2022.csv'):
    df_censo = pd.read_csv('data/censo_2022.csv')
else:
    df_censo = pd.DataFrame(columns=['cod_municipio'])

A tabela `cod_municipios` possui o código completo do IBGE com 7 dígitos na coluna `Código Município Completo`. 
Para fazer o join com a tabela do Bolsa Família (que usa 6 dígitos), precisaremos extrair os primeiros 6 dígitos.

In [3]:
df_cod['ibge_6'] = df_cod['Código Município Completo'].astype(str).str[:6].astype(int)

### 2. Tratamento da Base Bolsa Família
A tabela está em múltiplos meses (`anomes`). Agrupamos pelo código de município (`ibge`) usando a média para que a tabela fique apenas com os valores agregados por localidade.

In [4]:
df_bf_agg = df_bf.groupby('ibge').mean(numeric_only=True).reset_index()
df_bf_agg.head()

,ibge,anomes,qtd_ben_bas,qtd_ben_var,qtd_ben_bvj,qtd_ben_bvn,qtd_ben_bvg,qtd_ben_bsp
0,110001,202106.5,1143.4,2305.0,279.4,30.9,36.3,256.9
1,110002,202106.5,2913.6,6766.7,805.8,68.3,121.0,637.8
2,110003,202106.5,134.8,323.8,38.8,3.6,5.6,52.2
3,110004,202106.5,2536.1,5171.6,624.4,71.7,115.6,1083.6
4,110005,202106.5,494.8,1159.5,137.9,15.7,26.3,40.7


### 3. Join das Tabelas

Primeiro, criamos a tabela mapeada com `cod_municipios` onde adicionamos o código de 7 dígitos à base agregada do Bolsa Família.

In [5]:
df_bf_mapped = pd.merge(
    df_bf_agg, 
    df_cod[['ibge_6', 'Código Município Completo']], 
    left_on='ibge', 
    right_on='ibge_6', 
    how='inner'
)
df_bf_mapped.head()

,ibge,anomes,qtd_ben_bas,qtd_ben_var,qtd_ben_bvj,qtd_ben_bvn,qtd_ben_bvg,qtd_ben_bsp,ibge_6,Código Município Completo
0,110001,202106.5,1143.4,2305.0,279.4,30.9,36.3,256.9,110001,1100015
1,110002,202106.5,2913.6,6766.7,805.8,68.3,121.0,637.8,110002,1100023
2,110003,202106.5,134.8,323.8,38.8,3.6,5.6,52.2,110003,1100031
3,110004,202106.5,2536.1,5171.6,624.4,71.7,115.6,1083.6,110004,1100049
4,110005,202106.5,494.8,1159.5,137.9,15.7,26.3,40.7,110005,1100056


Em seguida, unimos usando o código de 7 com a base `sidra.csv` (e em seguida o `censo_2022.csv`).

In [6]:
df_final_temp = pd.merge(
    df_sidra, 
    df_bf_mapped, 
    left_on='cod_municipio', 
    right_on='Código Município Completo', 
    how='inner'
)

df_final = pd.merge(
    df_final_temp,
    df_censo.drop(columns=['municipio'], errors='ignore'),
    on='cod_municipio',
    how='left'
)

# Limpando colunas duplicadas que foram usadas apenas para os cruzamentos
df_final.drop(columns=['ibge_6', 'Código Município Completo', 'anomes'], inplace=True, errors='ignore')
df_final.head()

,cod_municipio,municipio,populacao_total,pib_municipal,va_adm_publica,va_agropecuaria,va_industria,ibge,qtd_ben_bas,qtd_ben_var,qtd_ben_bvj,qtd_ben_bvn,qtd_ben_bvg,qtd_ben_bsp,taxa_alfabetizacao,populacao_urbana_2010
0,1100015,Alta Floresta D'Oeste - RO,22516.0,734467,173122,311469,27845,110001,1143.4,2305.0,279.4,30.9,36.3,256.9,91.59,13970.0
1,1100023,Ariquemes - RO,111148.0,3211294,782306,293001,407675,110002,2913.6,6766.7,805.8,68.3,121.0,637.8,94.08,76525.0
2,1100031,Cabixi - RO,5067.0,238414,46579,136659,8117,110003,134.8,323.8,38.8,3.6,5.6,52.2,89.82,2693.0
3,1100049,Cacoal - RO,86416.0,2792506,629109,341315,236801,110004,2536.1,5171.6,624.4,71.7,115.6,1083.6,93.71,61921.0
4,1100056,Cerejeiras - RO,16088.0,743062,123365,151690,27433,110005,494.8,1159.5,137.9,15.7,26.3,40.7,92.15,14419.0


In [7]:
print("Shape Final:", df_final.shape)
if 'taxa_alfabetizacao' in df_final.columns:
    print("Nulos em Alfabetização:", df_final['taxa_alfabetizacao'].isna().sum())
if 'num_empresas' in df_final.columns:
    print("Nulos em Num Empresas:", df_final['num_empresas'].isna().sum())

Shape Final: (5570, 16)
Nulos em Alfabetização: 0


### 4. Exportar a Base Final

In [8]:
df_final.to_csv('data/base_final.csv', index=False)
print("Arquivo 'data/base_final.csv' formatado e salvo com sucesso!")

Arquivo 'data/base_final.csv' formatado e salvo com sucesso!
